### portmy database: stocks table
### stock database: buy, price tables

In [1]:
import pandas as pd
import numpy as np
from datetime import date, timedelta
from sqlalchemy import create_engine
from pandas.tseries.offsets import BDay

engine = create_engine("sqlite:///c:\\ruby\\portmy\\db\\development.sqlite3")
conmy = engine.connect()
conmy = engine.connect()
engine = create_engine("mysql+pymysql://root:@localhost:3306/stock")
const = engine.connect()

pd.set_option('display.float_format','{:,.2f}'.format)

data_path = "../data/"
csv_path = "\\Users\\User\\iCloudDrive\\"
box_path = "\\Users\\User\\Dropbox\\"

today = date.today()
today

datetime.date(2023, 1, 7)

### Process after last business day, yesterday must be last business day

In [2]:
# convert the timedelta object to a BusinessDay object
num_business_days = BDay(1)
yesterday = today - num_business_days
print(f'today: {today}')
print(f'yesterday: {yesterday}')

today: 2023-01-07
yesterday: 2023-01-06 00:00:00


In [3]:
yesterday = yesterday.date()
week_ago = yesterday -timedelta(days = 7)
print(f'a week ago: {week_ago}')
print(f'yesterday: {yesterday}')

a week ago: 2022-12-30
yesterday: 2023-01-06


### Restart & Run All Cells

### This process affects only already in port stocks. To highlight price changes in percent.

In [4]:
cols = 'name price_w price_d percent perform mrkt '.split()

format_dict = {
    'qty':'{:,}',    
    'price':'{:.2f}','price_d':'{:.2f}','price_w':'{:.2f}',
    'max_price':'{:.2f}','min_price':'{:.2f}',
    'maxp':'{:.2f}','minp':'{:.2f}','opnp':'{:.2f}',    
    'pct':'{:,.2f}%','percent':'{:,.2f}%',
    
    'pe':'{:.2f}','pbv':'{:.2f}',
    'paid_up':'{:,.2f}','market_cap':'{:,.2f}',
    'daily_volume':'{:,.2f}','beta':'{:,.2f}', 
    'dly_vol':'{:,.2f}', 
}

In [5]:
sql = """
SELECT name, date, price 
FROM buy 
ORDER BY date DESC 
LIMIT 1
"""
df_buy = pd.read_sql(sql, const)
df_buy.style.format(format_dict)

,name,date,price
0,BANPU,2023-01-06,12.30


In [6]:
names = df_buy['name']
name = names.to_string(index=False)

sql = '''
SELECT * FROM stocks WHERE name = '%s'
'''
sql = sql % name
print(sql)

df_stocks = pd.read_sql(sql, conmy)
df_stocks.style.format(format_dict) 


SELECT * FROM stocks WHERE name = 'BANPU'



,id,name,market,price,max_price,min_price,pe,pbv,paid_up,market_cap,daily_volume,beta,ticker_id,created_at,updated_at
0,47,BANPU,SET50 / SETCLMV / SETTHSI,12.30,15.00,10.50,2.35,0.93,"8,454.16","103,986.19","2,734.46",0.73,47,2017-07-23 07:24:25.883582,2023-01-06 21:29:43.950540


In [7]:
sql = '''
SELECT * FROM price WHERE name = "%s" ORDER BY date DESC LIMIT 5
'''
sql = sql % name
print(sql)

df_price = pd.read_sql(sql, const)
df_price.style.format(format_dict)


SELECT * FROM price WHERE name = "BANPU" ORDER BY date DESC LIMIT 5



,name,date,price,maxp,minp,qty,opnp
0,BANPU,2023-01-06,12.30,12.80,12.30,"294,714,032",12.70
1,BANPU,2023-01-05,12.80,12.90,12.70,"150,146,802",12.80
2,BANPU,2023-01-04,12.80,13.40,12.80,"305,436,030",13.30
3,BANPU,2023-01-03,13.40,13.70,13.40,"104,091,420",13.70
4,BANPU,2022-12-30,13.70,13.80,13.60,"49,373,887",13.70


In [8]:
sql = '''
SELECT name
FROM buy 
WHERE active = 1
ORDER BY name'''
df_price = pd.read_sql(sql, const)

names = df_price.name.tolist()
in_p = ", ".join(map(lambda name: "'%s'" % name, names))

sql = """
SELECT name, price AS price_d 
FROM price 
WHERE date = '%s' AND name IN (%s)
ORDER BY name, date"""
sql = sql % (yesterday, in_p)
print(sql)

df_today = pd.read_sql(sql, const)
df_today.shape[0]


SELECT name, price AS price_d 
FROM price 
WHERE date = '2023-01-06' AND name IN ('ASP', 'BANPU', 'BCH', 'CPNREIT', 'DCC', 'DIF', 'GVREIT', 'IVL', 'JASIF', 'JMART', 'KCE', 'MAKRO', 'MCS', 'NER', 'ORI', 'PTTGC', 'RCL', 'SCC', 'SCCC', 'SENA', 'SINGER', 'STA', 'SYNEX', 'TFFIF', 'TMT', 'WHAIR', 'WHART')
ORDER BY name, date


27

In [9]:
sql = """
SELECT name, price AS price_w
FROM price 
WHERE date = '%s' AND name IN (%s) 
ORDER BY name"""
sql = sql % (week_ago, in_p)
print(sql)

df_wkago = pd.read_sql(sql, const)
df_wkago.shape[0]


SELECT name, price AS price_w
FROM price 
WHERE date = '2022-12-30' AND name IN ('ASP', 'BANPU', 'BCH', 'CPNREIT', 'DCC', 'DIF', 'GVREIT', 'IVL', 'JASIF', 'JMART', 'KCE', 'MAKRO', 'MCS', 'NER', 'ORI', 'PTTGC', 'RCL', 'SCC', 'SCCC', 'SENA', 'SINGER', 'STA', 'SYNEX', 'TFFIF', 'TMT', 'WHAIR', 'WHART') 
ORDER BY name


27

In [10]:
trend = pd.merge(df_today, df_wkago, on='name',how='inner')
trend.shape

(27, 3)

In [11]:
def performance(vals):
    price_d, price_w = vals
    if (price_d > price_w):
        return 'Better'
    elif (price_d < price_w):
        return 'Worse'
    else:
        return 'No Change'

In [12]:
trend['percent'] = (trend.price_d-trend.price_w)/trend.price_w * 100

In [13]:
trend

,name,price_d,price_w,percent
0,ASP,2.98,2.98,0.00
1,BANPU,12.30,13.70,-10.22
2,BCH,21.90,20.50,6.83
3,CPNREIT,19.50,19.50,0.00
4,DCC,2.84,2.82,0.71
5,DIF,13.20,13.20,0.00
6,GVREIT,9.00,9.10,-1.10
7,IVL,37.75,40.75,-7.36
8,JASIF,8.15,8.05,1.24
9,JMART,40.25,40.75,-1.23


In [14]:
trend["perform"] = trend[["price_d","price_w"]].apply(performance, axis=1)

In [15]:
trend.sort_values(by=['percent'],ascending=[True]).style.format(format_dict)

,name,price_d,price_w,percent,perform
1,BANPU,12.30,13.70,-10.22%,Worse
7,IVL,37.75,40.75,-7.36%,Worse
20,SINGER,27.00,28.75,-6.09%,Worse
14,ORI,11.60,12.10,-4.13%,Worse
15,PTTGC,46.25,47.25,-2.12%,Worse
24,TMT,7.70,7.80,-1.28%,Worse
9,JMART,40.25,40.75,-1.23%,Worse
6,GVREIT,9.00,9.10,-1.10%,Worse
16,RCL,30.50,30.75,-0.81%,Worse
13,NER,6.30,6.35,-0.79%,Worse


### Filter only performance = "Worse"

In [16]:
mask = trend.perform == 'Worse'
trend[mask]

,name,price_d,price_w,percent,perform
1,BANPU,12.30,13.70,-10.22,Worse
6,GVREIT,9.00,9.10,-1.10,Worse
7,IVL,37.75,40.75,-7.36,Worse
9,JMART,40.25,40.75,-1.23,Worse
13,NER,6.30,6.35,-0.79,Worse
14,ORI,11.60,12.10,-4.13,Worse
15,PTTGC,46.25,47.25,-2.12,Worse
16,RCL,30.50,30.75,-0.81,Worse
19,SENA,3.92,3.94,-0.51,Worse
20,SINGER,27.00,28.75,-6.09,Worse


In [17]:
trend.perform.value_counts(normalize=True).to_frame().style.format('{:.2%}')

,perform
Worse,40.74%
Better,40.74%
No Change,18.52%


In [18]:
sql = '''
SELECT name, max_price AS max, min_price AS min, pe, pbv, daily_volume AS volume, beta, market
FROM stocks
'''
my_stocks = pd.read_sql(sql, conmy)
my_stocks.shape

(253, 8)

In [19]:
filters = [
   (my_stocks.market.str.contains('SET50')),
   (my_stocks.market.str.contains('SET100')),
   (my_stocks.market.str.contains('mai'))]
values = ['SET50','SET100','mai']
my_stocks["mrkt"] = np.select(filters, values, default='SET999')

In [20]:
trend2 = pd.merge(trend, my_stocks, on='name', how='inner')
trend2.sort_values(['percent'],ascending=[True]).shape

(27, 13)

In [21]:
set50 = trend2.mrkt.str.contains('SET50') 
flt_set50 = trend2[set50]
flt_set50[cols].sort_values(by=['percent','name'],ascending=[True,True]).style.format(format_dict)

,name,price_w,price_d,percent,perform,mrkt
1,BANPU,13.70,12.30,-10.22%,Worse,SET50
7,IVL,40.75,37.75,-7.36%,Worse,SET50
15,PTTGC,47.25,46.25,-2.12%,Worse,SET50
9,JMART,40.75,40.25,-1.23%,Worse,SET50
17,SCC,342.00,350.00,2.34%,Better,SET50


In [22]:
flt_set50.perform.value_counts(normalize=True).to_frame().style.format('{:.2%}')

,perform
Worse,80.00%
Better,20.00%


In [23]:
set100 = trend2.mrkt.str.contains('SET100')
flt_set100 = trend2[set100]
flt_set100[cols].sort_values(by=['percent','name'],ascending=[True,True]).style.format(format_dict)

,name,price_w,price_d,percent,perform,mrkt
20,SINGER,28.75,27.00,-6.09%,Worse,SET100
14,ORI,12.10,11.60,-4.13%,Worse,SET100
16,RCL,30.75,30.50,-0.81%,Worse,SET100
21,STA,21.10,21.10,0.00%,No Change,SET100
10,KCE,46.50,48.00,3.23%,Better,SET100
2,BCH,20.50,21.90,6.83%,Better,SET100


In [24]:
flt_set100[cols].perform.value_counts(normalize=True).to_frame().style.format('{:.2%}')

,perform
Worse,50.00%
Better,33.33%
No Change,16.67%


In [25]:
set999 = trend2.mrkt.str.contains('SET999')
flt_set999 = trend2[set999]
flt_set999[cols].sort_values(by=['percent','name'],ascending=[True,True]).style.format(format_dict)

,name,price_w,price_d,percent,perform,mrkt
24,TMT,7.80,7.70,-1.28%,Worse,SET999
6,GVREIT,9.10,9.00,-1.10%,Worse,SET999
13,NER,6.35,6.30,-0.79%,Worse,SET999
19,SENA,3.94,3.92,-0.51%,Worse,SET999
0,ASP,2.98,2.98,0.00%,No Change,SET999
3,CPNREIT,19.50,19.50,0.00%,No Change,SET999
5,DIF,13.20,13.20,0.00%,No Change,SET999
26,WHART,10.70,10.70,0.00%,No Change,SET999
23,TFFIF,7.65,7.70,0.65%,Better,SET999
25,WHAIR,7.45,7.50,0.67%,Better,SET999


In [26]:
flt_set999[cols].perform.value_counts(normalize=True).to_frame().style.format('{:.2%}')

,perform
Better,50.00%
No Change,25.00%
Worse,25.00%


In [27]:
setmai = trend2.mrkt.str.contains('mai')
flt_setmai = trend2[setmai]
flt_setmai[cols].sort_values(by=['percent','name'],ascending=[True,True]).style.format(format_dict)

,name,price_w,price_d,percent,perform,mrkt


In [28]:
flt_setmai[cols].perform.value_counts(normalize=True).to_frame().style.format('{:.2%}')

,perform
